## Tracking Data: Extracting multiple frames per shot

First lets create a function for extracting all the videoTimestamp from all the shots given a match

In [6]:
import json
import os

def extract_video_timestamps_from_shot_file(filepath):
    file_key = filepath.split("/")[-1].replace(".json", "")
    with open(filepath, 'r') as f:
        data = json.load(f)
    timestamps = [event['videoTimestamp'] for event in data if 'videoTimestamp' in event]
    return {file_key: timestamps}



In [7]:
df = extract_video_timestamps_from_shot_file("shot_pack/shots/644507.json")
df

{'644507': ['394.419335',
  '504.904709',
  '590.015496',
  '685.013836',
  '1177.727182',
  '1279.560962',
  '1510.875063',
  '2090.917203',
  '2344.396955',
  '2460.344738',
  '2788.209796',
  '2849.860245',
  '2907.556667',
  '3474.098645',
  '3727.147296',
  '3728.614645',
  '4227.847976',
  '4345.074861',
  '4357.240709',
  '4539.47338',
  '4541.568165',
  '5009.651231',
  '5011.346379',
  '5122.164576',
  '5452.536857',
  '5533.427781']}

Let's also extract the assists videoTimestamp

In [8]:
def extract_assist_info_video_timestamps(filepath):
    file_key = filepath.split("/")[-1].replace(".json", "")
    with open(filepath, 'r') as f:
        data = json.load(f)
    
    timestamps = [
        event['assist_info']['videoTimestamp']
        for event in data
        if event.get('assist_info') and 'videoTimestamp' in event['assist_info']
    ]
    return {file_key: timestamps}

In [9]:
df2 = extract_assist_info_video_timestamps("shot_pack/shots/644507.json")
df2

{'644507': ['392.154206',
  '498.810584',
  '589.109165',
  '676.770239',
  '1176.908849',
  '1278.425758',
  '1510.501844',
  '2088.648279',
  '2342.922265',
  '2458.617965',
  '2847.217446',
  '2905.588464',
  '3472.471286',
  '3722.439072',
  '4225.497118',
  '4343.089214',
  '4356.258991',
  '4536.421677',
  '5008.863938',
  '5008.863938',
  '5121.166837',
  '5451.42945',
  '5531.052243']}

Create a function that gets all frames from tracking data that have the closest videotimestamp from the shots. We can specify 
how many frames do we want from before and after the shot frame

In [10]:
def get_closest_tracking_frames_for_shots_with_context(shot_file, tracking_file, prev_frames=2, next_frames=2):
    """
    Extract closest tracking frames for shot timestamps, including previous and next frames around each match.
    
    Args:
        shot_file: Path to the shot JSON file
        tracking_file: Path to the tracking JSONL file
        prev_frames: Number of frames to include before the closest match (default: 2)
        next_frames: Number of frames to include after the closest match (default: 2)
    
    Returns:
        Dictionary with shot timestamps as keys and frame context data as values
    """
    with open(shot_file, 'r') as sf:
        shot_data = json.load(sf)
    
    video_timestamps = [
        str(event['videoTimestamp']) for event in shot_data 
        if 'videoTimestamp' in event and event.get('type', {}).get('primary') == 'shot'
    ]
    
    # First pass: read all tracking frames into memory
    tracking_frames = []
    with open(tracking_file, 'r') as tf:
        for line in tf:
            try:
                frame = json.loads(line)
                ts = frame.get('videoTimestamp') or frame.get('Videotimestamp')
                if ts is not None:
                    tracking_frames.append(frame)
            except json.JSONDecodeError:
                continue
    
    # Sort tracking frames by timestamp for easier context extraction
    tracking_frames.sort(key=lambda x: x.get('videoTimestamp') or x.get('Videotimestamp'))
    
    closest_frames_with_context = {}
    
    # Second pass: find closest matches and extract context
    for shot_ts in video_timestamps:
        shot_timestamp = float(shot_ts)
        min_diff = float('inf')
        closest_index = -1
        
        # Find the closest frame index
        for i, frame in enumerate(tracking_frames):
            frame_ts = frame.get('videoTimestamp') or frame.get('Videotimestamp')
            diff = abs(frame_ts - shot_timestamp)
            if diff < min_diff:
                min_diff = diff
                closest_index = i
        
        if closest_index != -1:
            # Extract previous frames (if available)
            prev_start_idx = max(0, closest_index - prev_frames)
            prev_frames_data = []
            for i in range(prev_start_idx, closest_index):
                frame_info = {
                    'frame': tracking_frames[i],
                    'position_relative_to_closest': i - closest_index,
                    'frame_type': 'previous'
                }
                prev_frames_data.append(frame_info)
            
            # Extract next frames (if available)  
            next_end_idx = min(len(tracking_frames), closest_index + next_frames + 1)
            next_frames_data = []
            for i in range(closest_index + 1, next_end_idx):
                frame_info = {
                    'frame': tracking_frames[i],
                    'position_relative_to_closest': i - closest_index,
                    'frame_type': 'next'
                }
                next_frames_data.append(frame_info)
            
            # Combine all frames for backward compatibility
            all_context_frames = prev_frames_data + [{
                'frame': tracking_frames[closest_index],
                'position_relative_to_closest': 0,
                'frame_type': 'closest',
                'is_closest': True
            }] + next_frames_data
            
            closest_frames_with_context[shot_ts] = {
                'closest_frame': tracking_frames[closest_index],
                'previous_frames': prev_frames_data,
                'next_frames': next_frames_data
            }
    
    return closest_frames_with_context

And create a function to itarate from all the files

In [11]:
def get_all_shot_tracking_matches_with_context(shots_dir, tracking_dir, prev_frames=2, next_frames=2, max_files=10):
    """
    Process shot-tracking matches with context frames (limited to first max_files).
    
    Args:
        shots_dir: Directory containing shot JSON files
        tracking_dir: Directory containing tracking JSONL files
        prev_frames: Number of frames to include before each closest match (default: 2)
        next_frames: Number of frames to include after each closest match (default: 2)
        max_files: Maximum number of files to process (default: 10)
    
    Returns:
        Dictionary with match IDs as keys and frame context data as values
    """
    all_matches = {}
    processed_count = 0
    
    # Get sorted list of JSON files to ensure consistent ordering
    json_files = [f for f in os.listdir(shots_dir) if f.endswith('.json')]
    json_files.sort()  # Sort for consistent processing order
    
    for filename in json_files:
        if processed_count >= max_files:
            print(f"Reached maximum file limit ({max_files}). Stopping processing.")
            break
            
        match_id = filename.replace('.json', '')
        shot_path = os.path.join(shots_dir, f"{match_id}.json")
        tracking_path = os.path.join(tracking_dir, f"{match_id}_tracking_data.jsonl")
        
        if os.path.exists(tracking_path):
            result = get_closest_tracking_frames_for_shots_with_context(
                shot_path, tracking_path, prev_frames, next_frames
            )
            all_matches[match_id] = result
            processed_count += 1
            
            print(f"Processed {processed_count}/{max_files}: {match_id}")
        else:
            print(f"Tracking file missing for {match_id}")
    
    print(f"Total matches processed: {processed_count} out of {max_files} requested")
    return all_matches

In [12]:
all_data = get_all_shot_tracking_matches_with_context("shot_pack/shots", "shot_pack/jsonls",5,5)

Processed 1/10: 644494
Processed 2/10: 644507
Processed 3/10: 644510
Processed 4/10: 644513
Processed 5/10: 644524
Processed 6/10: 644543
Processed 7/10: 644545
Processed 8/10: 644546
Processed 9/10: 644547
Processed 10/10: 644548
Reached maximum file limit (10). Stopping processing.
Total matches processed: 10 out of 10 requested


In [13]:
all_data["644507"]["394.419335"].keys()

dict_keys(['closest_frame', 'previous_frames', 'next_frames'])

Lets create also a function to extract the match info

In [14]:
def extract_match_data_from_directory(directory_path):
    """
    Create a dictionary where keys are match_ids and values are match_data
    from all JSON files in the specified directory.
    
    Args:
        directory_path (str): Path to the directory containing the JSON files
        
    Returns:
        dict: Dictionary with match_id as keys and match_data as values
    """
    match_data_dict = {}
    
    # Check if directory exists
    if not os.path.exists(directory_path):
        raise FileNotFoundError(f"Directory '{directory_path}' does not exist")
    
    # Iterate through all files in the directory
    for filename in os.listdir(directory_path):
        file_path = os.path.join(directory_path, filename)
        
        # Skip if it's not a file
        if not os.path.isfile(file_path):
            continue
            
        try:
            with open(file_path, 'r', encoding='utf-8') as file:
                # Read the first line which contains the match_data
                first_line = file.readline().strip()
                
                if first_line:
                    # Parse the JSON from the first line
                    data = json.loads(first_line)
                    
                    # Extract all the data (match_data, teams_data, players_data, FPS)
                    if 'match_data' in data:
                        match_id = data['match_data'].get('match_id')
                        
                        if match_id:
                            # Convert match_id to string to ensure consistency
                            match_id_str = str(match_id)
                            # Store the entire first line data (not just match_data)
                            match_data_dict[match_id_str] = data
                        else:
                            print(f"Warning: No match_id found in file '{filename}'")
                    else:
                        print(f"Warning: No match_data found in file '{filename}'")
                        
        except json.JSONDecodeError as e:
            print(f"Error parsing JSON in file '{filename}': {e}")
        except Exception as e:
            print(f"Error reading file '{filename}': {e}")
    
    return match_data_dict

In [15]:
match_info= extract_match_data_from_directory("shot_pack/jsonls")
match_info.keys()

dict_keys(['644645', '644614', '670136', '644567', '662713', '644594', '652706', '670964', '670137', '644644', '644615', '644803', '644595', '662712', '644566', '652704', '644565', '644596', '662711', '662708', '644800', '644616', '644647', '644792', '662710', '644597', '644564', '652705', '644617', '644646', '662709', '644608', '644659', '644562', '644591', '662716', '644588', '670960', '644640', '644590', '644563', '644609', '644658', '644641', '644794', '644806', '670961', '644589', '644797', '644642', '644613', '670962', '644560', '662714', '644593', '652701', '670963', '644578', '644643', '644796', '644612', '671443', '652700', '644592', '662715', '644561', '644606', '644657', '644810', '662718', '644586', '644575', '644524', '662719', '644607', '644656', '644738', '644574', '644587', '644799', '644584', '644655', '644604', '644576', '644585', '644798', '644654', '644605', '644583', '670138', '644570', '644652', '644603', '673373', '670139', '644653', '644650', '644817', '644598',

In [16]:
match_info["644645"]

{'match_data': {'season_data': {'name': 'UCL 2024', 'id': 818378},
  'date': '2024-12-11',
  'match_id': 644645,
  'result': {'home': 0, 'away': 0}},
 'teams_data': {'home': {'name': 'SL Benfica', 'id': 61318},
  'away': {'name': 'Bologna', 'id': 12}},
 'players_data': {'61318': {'1262779': {'number': 1,
    'name': 'Anatolii Trubin',
    'position': 'GK'},
   '1267199': {'number': 3, 'name': 'Alvaro Fernandez', 'position': 'DL'},
   '1528624': {'number': 6, 'name': 'Alexander Bah', 'position': 'DR'},
   '122541': {'number': 30, 'name': 'Nicolas Otamendi', 'position': 'DC'},
   '1339424': {'number': 44, 'name': 'Tomas Araujo', 'position': 'DC'},
   '1156661': {'number': 8, 'name': 'Fredrik Aursnes', 'position': 'MC'},
   '1172508': {'number': 10, 'name': 'Orkun Kokcu', 'position': 'MC'},
   '1261490': {'number': 61, 'name': 'Florentino', 'position': 'DMC'},
   '80742': {'number': 11, 'name': 'Angel Di Maria', 'position': 'FWR'},
   '1158780': {'number': 14, 'name': 'Vangelis Pavlidis',

### VISUALIZATIONS

A function that plots all the frames for a specific shot

In [17]:
import matplotlib.pyplot as plt
from mplsoccer import Pitch

def plot_all_frames_for_shot(shot_timestamp_data, match_data):
    """
    Plot all frames (2 previous, closest, 2 next) for a single shot event.
    
    Args:
        shot_timestamp_data: Dictionary containing 'previous_frames', 'closest_frame', 'next_frames'
        match_data: Dictionary containing complete match data for a specific match
    """
    # Collect all frames in chronological order
    all_frames = []
    frame_types = []
    
    # Add previous frames
    for prev_frame in shot_timestamp_data['previous_frames']:
        all_frames.append(prev_frame['frame'])
        frame_types.append(f"Previous ({prev_frame['position_relative_to_closest']:+d})")
    
    # Add closest frame
    all_frames.append(shot_timestamp_data['closest_frame'])
    frame_types.append("CLOSEST (Shot)")
    
    # Add next frames
    for next_frame in shot_timestamp_data['next_frames']:
        all_frames.append(next_frame['frame'])
        frame_types.append(f"Next ({next_frame['position_relative_to_closest']:+d})")
    
    # Create subplots for all frames
    num_frames = len(all_frames)
    if num_frames == 0:
        print("No frames to plot")
        return
    
    # Get match info
    home_team = match_data['teams_data']['home']['name']
    away_team = match_data['teams_data']['away']['name']
    season = match_data['match_data']['season_data']['name']
    match_title = f"{home_team} vs {away_team} {season}"
    
    # Get team IDs and player data for position lookup
    home_team_id = str(match_data['teams_data']['home']['id'])
    away_team_id = str(match_data['teams_data']['away']['id'])
    players_data = match_data['players_data']
    
    # Calculate subplot layout (prefer horizontal layout for up to 5 frames)
    if num_frames <= 3:
        rows, cols = 1, num_frames
        figsize = (6 * num_frames, 6)
    else:
        rows, cols = 2, (num_frames + 1) // 2
        figsize = (6 * cols, 6 * rows)
    
    fig, axes = plt.subplots(rows, cols, figsize=figsize)
    
    # Set main title for the entire figure
    fig.suptitle(match_title, fontsize=16, fontweight='bold', y=0.98)
    
    if num_frames == 1:
        axes = [axes]
    elif rows == 1:
        axes = axes
    else:
        axes = axes.flatten()
    
    # Plot each frame
    for i, (frame, frame_type) in enumerate(zip(all_frames, frame_types)):
        ax = axes[i]
        
        # Extract frame data
        team_ids = list(frame['data'].keys())
        if len(team_ids) != 2:
            ax.text(0.5, 0.5, f"Invalid frame data\n{frame_type}", 
                   ha='center', va='center', transform=ax.transAxes)
            continue
        
        team0_id, team1_id = team_ids
        team0 = frame['data'][team0_id]
        team1 = frame['data'][team1_id]
        ball = frame['ball']
        
        # Create pitch for this subplot
        pitch = Pitch(pitch_type='custom', pitch_length=105, pitch_width=68,
                      pitch_color='grass', line_color='white')
        pitch.draw(ax=ax)
        
        # Function to get player colors based on position
        def get_player_colors(team_players, team_id):
            colors = []
            for player in team_players:
                player_id = str(player['id'])
                if team_id in players_data and player_id in players_data[team_id]:
                    position = players_data[team_id][player_id]['position']
                    if position == 'GK':
                        colors.append('yellow')
                    else:
                        colors.append('blue' if team_id == home_team_id else 'red')
                else:
                    # Default color if player not found
                    colors.append('blue' if team_id == home_team_id else 'red')
            return colors
        
        # Get colors for each team
        team0_colors = get_player_colors(team0, team0_id)
        team1_colors = get_player_colors(team1, team1_id)
        
        # Plot players with appropriate colors
        pitch.scatter([p['x'] for p in team0], [p['y'] for p in team0], ax=ax,
                      c=team0_colors, s=80, edgecolors='black', alpha=0.8)
        pitch.scatter([p['x'] for p in team1], [p['y'] for p in team1], ax=ax,
                      c=team1_colors, s=80, edgecolors='black', alpha=0.8)
        
        # Plot ball if available
        if ball and ball[0] is not None and ball[1] is not None:
            pitch.scatter(ball[0], ball[1], ax=ax,
                          color='white', edgecolors='black', s=150)
        
        # Set title with frame info
        timestamp = frame.get('videoTimestamp') or frame.get('Videotimestamp')
        title_color = 'red' if 'CLOSEST' in frame_type else 'black'
        ax.set_title(f"{frame_type}\nTime: {timestamp:.2f}s", 
                    fontsize=12, fontweight='bold' if 'CLOSEST' in frame_type else 'normal',
                    color=title_color)
    
    # Hide unused subplots
    for i in range(num_frames, len(axes)):
        axes[i].set_visible(False)
    
    plt.tight_layout()
    plt.show()

In [18]:
all_data["644494"].keys()

dict_keys(['182.042218', '805.369128', '926.011795', '929.434655', '1365.006326', '1506.56981', '1664.561848', '2003.1784', '2128.990257', '2163.82168', '2399.772393', '2450.490236', '3447.066107', '3822.117414', '4325.487962', '5087.551377', '5217.118409', '5299.51628', '5346.50587', '5446.189353', '5675.219009'])

In [ ]:
plot_all_frames_for_shot(all_data["644494"]["926.011795"], match_info["644494"])

A function that plots all the frames for all the shots for a specific match

In [19]:
def plot_all_shots_from_match(match_results, match_data, match_id=None, max_shots=None):
    """
    Plot all frames for all shots in a match or across all matches.
    
    Args:
        match_results: Dictionary from get_all_shot_tracking_matches_with_context()
        match_id: Specific match ID to plot (if None, plots all matches)
        max_shots: Maximum number of shots to plot (if None, plots all shots)
    """
    if isinstance(match_results, dict) and 'closest_frame' in str(match_results):
        # Single match data passed directly
        shots_to_plot = match_results
        title_prefix = f"Match {match_id}" if match_id else "Match"
    else:
        # Multiple matches data
        if match_id and match_id in match_results:
            shots_to_plot = match_results[match_id]
            title_prefix = f"Match {match_id}"
        elif not match_id:
            # Plot from first match if no specific match requested
            first_match = next(iter(match_results.keys()))
            shots_to_plot = match_results[first_match]
            title_prefix = f"Match {first_match}"
        else:
            print(f"Match ID {match_id} not found in results")
            return
    
    shot_count = 0
    for shot_timestamp, shot_data in shots_to_plot.items():
        if max_shots and shot_count >= max_shots:
            break
        
        print(f"\n{title_prefix} - Shot at timestamp {shot_timestamp}")
        plot_all_frames_for_shot(shot_data,match_data)
        shot_count += 1


In [20]:
all_data.keys()

dict_keys(['644494', '644507', '644510', '644513', '644524', '644543', '644545', '644546', '644547', '644548'])

In [ ]:
plot_all_shots_from_match(all_data["644510"],match_info["644510"])

In [21]:
all_data.keys()

dict_keys(['644494', '644507', '644510', '644513', '644524', '644543', '644545', '644546', '644547', '644548'])

In [ ]:
plot_all_shots_from_match(all_data["644507"],match_info["644507"])

In [43]:
null_balls=0
total_frames=0
for match_id,_ in all_data.items():
    for _,frame in all_data[match_id].items():
        if frame["closest_frame"]["ball"]==[None, None]:
            null_balls+=1
        if frame["previous_frames"][0]["frame"]["ball"]==[None, None]:
            null_balls+=1
        if frame["previous_frames"][1]["frame"]["ball"]==[None, None]:
            null_balls+=1
        if frame["next_frames"][0]["frame"]["ball"]==[None, None]:
            null_balls+=1
        if frame["next_frames"][1]["frame"]["ball"]==[None, None]:
            null_balls+=1
        total_frames+=5  
print(f"Total  number of null_balls: {null_balls}")
print(f"Total  number of frames: {total_frames}")
print(f"Percentage of null_balls: {null_balls/total_frames}")


Total  number of null_balls: 704
Total  number of frames: 1240
Percentage of null_balls: 0.567741935483871


In [118]:
# 61,37% of frames have NULL ball
# GoalKeeper can be rescued but it is from other frames